# Brain Tumor MRI Classification — Step by Step

**Goal:** classify brain MRI images into four classes — **glioma**, **meningioma**, **no tumor** and **pituitary** tumor.

This notebook is written as a sequence of small steps. Each step has a short explanation of **what** it does and **why**, followed by the code. Run the cells from top to bottom.

**Part 1 (this notebook so far): data and preprocessing**

| Section | What happens |
|---|---|
| 1. Setup | import libraries, check the GPU, choose settings |
| 2. Load the data | download the dataset and list every image with its label |
| 3. Explore the data | class counts, sample images, image sizes |
| 4. Clean the data | find and remove duplicate images |
| 5. Split the data | training / validation / test sets |
| 6. Fix the class imbalance | oversample the smaller classes in the training set |
| 7. Data augmentation | create image generators that feed the models |

*Before starting on Google Colab: Runtime → Change runtime type → GPU.*

---
## 1. Setup

### Step 1.1 — Import libraries

| Library | Used for |
|---|---|
| `os`, `hashlib` | working with files and folders; creating a "fingerprint" of each image file |
| `numpy`, `pandas` | numbers and tables |
| `matplotlib` | charts and showing images |
| `PIL` (Pillow) | opening image files |
| `tensorflow` | building and training the neural networks |
| `ImageDataGenerator` | loading images in batches and applying data augmentation |
| `train_test_split` | splitting the data into training and validation sets |

In [ ]:
import os
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

### Step 1.2 — Check the GPU

Training neural networks on images is much faster on a GPU. This should print one GPU. If it prints an empty list, change the Colab runtime to GPU and restart.

In [ ]:
print("GPUs available:", tf.config.list_physical_devices("GPU"))

### Step 1.3 — Settings

These values are used throughout the notebook, so they are defined once here.

| Setting | Value | Meaning |
|---|---|---|
| `IMG_SIZE` | (224, 224) | every image is resized to 224 × 224 pixels — the standard input size for the models we will use |
| `BATCH_SIZE` | 32 | the number of images the model looks at in one training step |
| `SEED` | 42 | a fixed "random seed", so the random steps (splitting, shuffling) give the same result every time the notebook is run |

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

---
## 2. Load the data

### Step 2.1 — Download the dataset

We use the **Brain Tumor MRI Dataset** by Masoud Nickparvar on Kaggle. It combines three public collections (Figshare, SARTAJ and Br35H).

We download **version 1** (the original release). A later version 2 was made perfectly balanced by the publisher, but real hospital data is almost never balanced — some tumor types are much more common than others. Using version 1 lets us deal with class imbalance ourselves, the way we would have to with real data.

`kagglehub` downloads the dataset and returns the folder it was saved in.

In [ ]:
import kagglehub

download_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset/versions/1")
print("Downloaded to:", download_path)

### Step 2.2 — Find the Training and Testing folders

The dataset is already split by the publisher into a **Training** folder and a **Testing** folder. Inside each, there is one subfolder per class. This cell searches the downloaded folder for them.

In [ ]:
for root, folders, files in os.walk(download_path):
    if "Training" in folders and "Testing" in folders:
        DATASET_DIR = root
        break

TRAIN_DIR = os.path.join(DATASET_DIR, "Training")
TEST_DIR = os.path.join(DATASET_DIR, "Testing")

print("Training folder:", TRAIN_DIR)
print("Testing folder: ", TEST_DIR)
print("Class folders:  ", sorted(os.listdir(TRAIN_DIR)))

### Step 2.3 — Class names

The class names are the names of the subfolders. We sort them alphabetically so every part of the notebook uses the same order: glioma = 0, meningioma = 1, notumor = 2, pituitary = 3.

In [ ]:
CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)

### Step 2.4 — List every training image

We go through each class folder and record every image's **file path** and **label** (its class). The result is a table (a pandas DataFrame) with one row per image. Working with this table makes the next steps — counting, cleaning, splitting — simple.

In [ ]:
train_rows = []

for class_name in CLASS_NAMES:
    class_folder = os.path.join(TRAIN_DIR, class_name)
    for file_name in os.listdir(class_folder):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            train_rows.append({
                "image_path": os.path.join(class_folder, file_name),
                "label": class_name,
            })

train_full_df = pd.DataFrame(train_rows)
print("Training images:", len(train_full_df))
train_full_df.head()

### Step 2.5 — List every test image

The same for the Testing folder. The test set is kept aside and only used at the very end, to measure how well the final models work on images they have never seen.

In [ ]:
test_rows = []

for class_name in CLASS_NAMES:
    class_folder = os.path.join(TEST_DIR, class_name)
    for file_name in os.listdir(class_folder):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            test_rows.append({
                "image_path": os.path.join(class_folder, file_name),
                "label": class_name,
            })

test_df = pd.DataFrame(test_rows)
print("Test images:", len(test_df))
test_df.head()

---
## 3. Explore the data

### Step 3.1 — Number of images per class

A table of how many images each class has, in the training and test sets.

In [ ]:
class_counts = pd.DataFrame({
    "training": train_full_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
})
class_counts = class_counts.loc[CLASS_NAMES]
class_counts["total"] = class_counts["training"] + class_counts["test"]
class_counts

### Step 3.2 — Visualise the class distribution

The bar chart makes the imbalance easy to see: the classes do not have the same number of images. A model trained on imbalanced data tends to favour the bigger classes, so this is fixed in Section 6.

In [ ]:
class_counts[["training", "test"]].plot(kind="bar", figsize=(9, 5), rot=0, color=["#1f3864", "#7fa7d9"])
plt.title("Images per class")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.show()

largest = class_counts["training"].max()
smallest = class_counts["training"].min()
print(f"Largest training class:  {class_counts['training'].idxmax()} ({largest} images)")
print(f"Smallest training class: {class_counts['training'].idxmin()} ({smallest} images)")
print(f"Imbalance ratio: {largest / smallest:.2f}")

### Step 3.3 — Look at sample images

Four example images from each class, to get a feel for the data.

In [ ]:
plt.figure(figsize=(12, 12))

for row, class_name in enumerate(CLASS_NAMES):
    examples = train_full_df[train_full_df["label"] == class_name]["image_path"].head(4)
    for col, image_path in enumerate(examples):
        image = Image.open(image_path)
        plt.subplot(NUM_CLASSES, 4, row * 4 + col + 1)
        plt.imshow(image, cmap="gray")
        plt.title(class_name)
        plt.axis("off")

plt.tight_layout()
plt.show()

### Step 3.4 — Check the image sizes

The images come from three different collections, so they do not all have the same size. We check a random sample of 300 images. Because the sizes differ, every image will be resized to 224 × 224 in Section 7.

In [ ]:
sample_paths = train_full_df["image_path"].sample(300, random_state=SEED)

widths = []
heights = []
for image_path in sample_paths:
    width, height = Image.open(image_path).size
    widths.append(width)
    heights.append(height)

print("Width  — smallest:", min(widths), " largest:", max(widths))
print("Height — smallest:", min(heights), " largest:", max(heights))

---
## 4. Clean the data — remove duplicate images

### Step 4.1 — Create a fingerprint for every image

Because the dataset combines three collections, some images appear more than once — sometimes the **same image is in both the Training and Testing folders**. That is a problem: if the model has already seen a test image during training, its test score will look better than it really is (this is called **data leakage**).

To find duplicates we give each file a **fingerprint** (an MD5 hash). Two files with exactly the same content always get the same fingerprint, even if their file names are different. This takes a minute.

In [ ]:
def image_fingerprint(image_path):
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

train_full_df["fingerprint"] = train_full_df["image_path"].apply(image_fingerprint)
test_df["fingerprint"] = test_df["image_path"].apply(image_fingerprint)

train_full_df.head()

### Step 4.2 — Count the duplicates

Three checks:
1. training images that also appear in the test set (**data leakage**);
2. images repeated inside the training set;
3. images repeated inside the test set.

In [ ]:
test_fingerprints = set(test_df["fingerprint"])
in_test_set = train_full_df["fingerprint"].isin(test_fingerprints)

print("Training images that also appear in the test set:", in_test_set.sum())
print("Repeated images inside the training set:        ", train_full_df["fingerprint"].duplicated().sum())
print("Repeated images inside the test set:            ", test_df["fingerprint"].duplicated().sum())

### Step 4.3 — Remove the duplicates from the training set

- Training images that are also in the test set are **removed from training**. The test set stays exactly as the publisher released it.
- Repeated images inside the training set are kept **only once**.

In [ ]:
train_clean_df = train_full_df[~in_test_set]
train_clean_df = train_clean_df.drop_duplicates(subset="fingerprint")
train_clean_df = train_clean_df.reset_index(drop=True)

print("Training images before cleaning:", len(train_full_df))
print("Training images after cleaning: ", len(train_clean_df))
print("Removed:", len(train_full_df) - len(train_clean_df))

---
## 5. Split the data

### Step 5.1 — Create a validation set

We now have three sets, each with a different job:

| Set | Taken from | Used for |
|---|---|---|
| **Training** | 85% of the cleaned Training folder | the model learns from these images |
| **Validation** | 15% of the cleaned Training folder | checking progress during training, and choosing between models and settings |
| **Test** | the Testing folder | the final, independent score — used only at the very end |

`stratify` keeps the same class proportions in the training and validation sets.

In [ ]:
train_df, val_df = train_test_split(
    train_clean_df,
    test_size=0.15,
    stratify=train_clean_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training images:  ", len(train_df))
print("Validation images:", len(val_df))
print("Test images:      ", len(test_df))

### Step 5.2 — Class counts in each set

In [ ]:
split_counts = pd.DataFrame({
    "training": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
})
split_counts.loc[CLASS_NAMES]

---
## 6. Fix the class imbalance — oversampling

### Step 6.1 — Oversample the smaller classes

**The problem:** the classes have different numbers of training images. The model sees the bigger classes more often and may become biased towards them.

**The fix — oversampling:** for every class that is smaller than the biggest class, we randomly pick some of its images again until it has the same number as the biggest class. After this, every class has the same number of training images.

**Won't the model just see identical copies?** No. In Section 7 we add *data augmentation*: every time an image is loaded, it is randomly rotated, shifted, zoomed, flipped or brightened. So each repeated image looks slightly different to the model.

**Important:** oversampling is done **only on the training set**, and only **after** splitting. If we oversampled before splitting, copies of the same image could end up in both the training and validation sets, which would make the validation score unreliable. The validation and test sets are never changed.

In [ ]:
target_count = train_df["label"].value_counts().max()
print("Every class will have", target_count, "training images\n")

balanced_parts = []

for class_name in CLASS_NAMES:
    class_images = train_df[train_df["label"] == class_name]
    images_needed = target_count - len(class_images)

    extra_images = class_images.sample(n=images_needed, replace=True, random_state=SEED)
    balanced_parts.append(class_images)
    balanced_parts.append(extra_images)

    print(f"{class_name:<12} had {len(class_images):>5} images -> added {images_needed:>4} repeats")

train_balanced_df = pd.concat(balanced_parts)
train_balanced_df = train_balanced_df.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle

print("\nTraining images after oversampling:", len(train_balanced_df))

### Step 6.2 — Before and after oversampling

In [ ]:
before_after = pd.DataFrame({
    "before": train_df["label"].value_counts(),
    "after": train_balanced_df["label"].value_counts(),
}).loc[CLASS_NAMES]

before_after.plot(kind="bar", figsize=(9, 5), rot=0, color=["#94a3b8", "#1f3864"])
plt.title("Training images per class — before and after oversampling")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.show()

before_after

---
## 7. Data augmentation and image generators

### Step 7.1 — Define the augmentation for training images

An `ImageDataGenerator` loads images from disk in batches and prepares them for the model. For the **training** images it also applies random **data augmentation**:

| Setting | Value | What it does |
|---|---|---|
| `rescale` | 1/255 | converts pixel values from 0–255 to 0–1, which neural networks train better with |
| `rotation_range` | 10 | rotates the image by up to ±10 degrees |
| `width_shift_range`, `height_shift_range` | 0.05 | shifts the image left/right and up/down by up to 5% |
| `zoom_range` | 0.1 | zooms in or out by up to 10% |
| `horizontal_flip` | True | randomly mirrors the image left–right |
| `brightness_range` | 0.9 – 1.1 | makes the image up to 10% darker or brighter |
| `fill_mode`, `cval` | "constant", 0 | fills empty corners created by rotating/shifting with black, like the MRI background |

**Why these values are small:** the augmented images must still look like real brain scans. **Vertical flips are not used**, because brain scans always have the same top–bottom orientation.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    fill_mode="constant",
    cval=0,
)

### Step 7.2 — Define the preprocessing for validation and test images

Validation and test images are **only rescaled** (0–255 → 0–1), with **no augmentation**. They must stay unchanged so that they give an honest measure of performance.

In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

### Step 7.3 — Create the training generator

`flow_from_dataframe` reads the image paths and labels from our table (the oversampled training table) and produces batches of images:

- `target_size` — resize every image to 224 × 224;
- `color_mode="rgb"` — 3 colour channels, because the pretrained models expect colour input (the grey MRI value is simply copied into all three channels);
- `class_mode="categorical"` — each label becomes a one-hot vector, e.g. glioma = [1, 0, 0, 0];
- `classes=CLASS_NAMES` — fixes the class order so that it is the same in every generator;
- `shuffle=True` — mixes the order of images every epoch, so the model does not learn anything from the order.

In [ ]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_balanced_df,
    x_col="image_path",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    color_mode="rgb",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

### Step 7.4 — Create the validation generator

Same settings, but using the validation table, no augmentation, and `shuffle=False`. Keeping the original order means the model's predictions can later be matched to the correct labels.

In [ ]:
val_generator = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    color_mode="rgb",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

### Step 7.5 — Create the test generator

Same as the validation generator, using the test table.

In [ ]:
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    color_mode="rgb",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

### Step 7.6 — Check the class numbering

All three generators must use the same numbering for the classes.

In [ ]:
print("Training:  ", train_generator.class_indices)
print("Validation:", val_generator.class_indices)
print("Test:      ", test_generator.class_indices)

### Step 7.7 — See the augmentation in action

The top image is one original training image. The bottom row shows five random augmented versions of the **same** image — this is how the repeated images from the oversampling step look different to the model.

In [ ]:
example_path = train_df["image_path"].iloc[0]
example_image = Image.open(example_path).convert("RGB").resize(IMG_SIZE)
example_array = np.array(example_image, dtype="float32")

plt.figure(figsize=(15, 7))

plt.subplot(2, 5, 3)
plt.imshow(example_image)
plt.title("Original")
plt.axis("off")

for i in range(5):
    augmented = train_datagen.random_transform(example_array, seed=SEED + i)
    augmented = train_datagen.standardize(augmented.copy())
    plt.subplot(2, 5, 6 + i)
    plt.imshow(np.clip(augmented, 0, 1))
    plt.title(f"Augmented #{i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

### Step 7.8 — Check one batch

A final check that the generator produces what the models need:

- **images:** shape `(32, 224, 224, 3)` — 32 images of 224 × 224 pixels with 3 colour channels;
- **labels:** shape `(32, 4)` — one one-hot label per image;
- **pixel values:** between 0 and 1.

In [ ]:
images, labels = next(train_generator)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Pixel values from", images.min(), "to", images.max())

plt.figure(figsize=(15, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])
    plt.title(CLASS_NAMES[np.argmax(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

train_generator.reset()

---
## Preprocessing complete

The data is now ready for training:

| Ready for the models | Contents |
|---|---|
| `train_generator` | oversampled, balanced training images with random augmentation |
| `val_generator` | validation images, rescaled only |
| `test_generator` | test images, rescaled only — kept for the final evaluation |

**Next part:** training and comparing the models — pretrained models (MobileNetV2, EfficientNetB0, Xception) with fine-tuning, and models trained from scratch (a custom CNN and Fibonacci-Net).